# Verify native categorical pilot and retain initial transport diagnostic


Verify all real-data pilot hashes, frozen split/source exclusions and primitive parity. The target remains prediction-grade, not independently admitted biological truth.

In [1]:
from pathlib import Path
import json, hashlib, ast
import numpy as np
import pandas as pd
from starplast import artifacts as A, label_records as L, strategies as S, organisms as O
from starplast.splits import read_split
out = Path('results/label_knn_pilot_2026_10_08_v2')
m = json.loads((out / 'manifest.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for path, digest in m['input_sha256'].items():
    assert sha(path) == digest, path
for name, digest in m['outputs'].items():
    assert sha(out / name) == digest, name
split, exclusions = read_split(out / 'split.json')
raw = json.loads((out / 'held_out/manifest.json').read_text())
spec = A.spec_from_dict(raw['spec'])
artifact = A.read_artifact(out / 'held_out', expected=spec, split=split)
state = artifact.payloads['model_state.json']
frame = pd.read_parquet(O.nodes_path(O.TOXOPLASMA)).set_index('gene_id')
ids = tuple(a.entity for a in split.assignments)
features = frame.loc[list(ids), state['input_columns']]
labels = pd.Series(state['training_labels'], index=state['fit_entities'])
batch = L.feature_knn(features, labels, split=split, exclusions=exclusions, k=state['k'], min_share=state['min_share'])
assert json.loads(batch.rows.to_json(orient='records')) == [{k: row[k] for k in batch.rows.columns} for row in artifact.payloads['rows.json']]
np.testing.assert_allclose(batch.class_scores.to_numpy(), np.array(artifact.payloads['class_scores.json']['scores'], dtype=float))
assert len(batch.rows) == 560 and not m['summary']['benchmark_admitted']
for path in ('starplast/label_records.py', 'scripts/freeze_label_knn_pilot.py'):
    ast.parse(Path(path).read_text(), feature_version=(3, 10))
{'retained_outer_test_genes': len(batch.rows), 'native_replay': 'exact', 'grade': m['summary']['truth_grade'], 'biological_admission': False}

{'retained_outer_test_genes': 560, 'native_replay': 'exact', 'grade': 'prediction', 'biological_admission': False}

The initial attempt completed fitting and wrote a valid child artifact, then failed a Python-container equality assertion because JSON converts tuple metadata into lists. The corrected script explicitly normalizes transport containers. Its model math is unchanged; the original child bytes and script remain available for inspection.

In [2]:
diagnostic = Path('results/label_knn_pilot_2026_10_08')
old_raw = json.loads((diagnostic / 'held_out/manifest.json').read_text())
old_spec = A.spec_from_dict(old_raw['spec'])
old = A.read_artifact(diagnostic / 'held_out', expected=old_spec, split=split)
assert old.payloads == artifact.payloads
{'diagnostic_model_and_payloads': 'identical', 'failure': 'Python tuple/list container comparison after JSON transport; corrected', 'original_script_sha256': sha(diagnostic / 'code/freeze_label_knn_pilot.py')}

{'diagnostic_model_and_payloads': 'identical', 'failure': 'Python tuple/list container comparison after JSON transport; corrected', 'original_script_sha256': 'c32465f828b97e8c3742381a1396a96979f188ac772ac27f7f02daaffb74aadd'}